# Phase 2 — Data Processing (Days 2-3)

**Goal:** (1) Tile every downloaded .svs slide into 512×512 patches with Reinhard
normalization. (2) Compute RNA-seq ground-truth labels: CD274 expression, immune
gene set scores, immune phenotype, immune score.

**Outputs:**
- `data/processed/patches/{slide_id}/` — 512×512 JPEG patches per slide
- `data/signatures/immune_signatures.csv` — Per-patient immune labels

---
**Hard Rules:**
- Patch size: **512×512** at 20× magnification (~0.5 µm/px)
- Stain normalization: **Reinhard** (LAB color space, NOT Macenko)
- Tissue threshold: **≥50%** tissue content per patch
- CD274 expression: log2(TPM+1) → **median split** → high/low
- TME subtypes: IE, IE/F, F, D (slash notation)
- MSI-L → treated as MSS (clinical convention)

## Colab setup

In [1]:


import os

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"

# Create processing output directories
for d in [
    f"{DATA_DIR}/processed/patches",
    f"{DATA_DIR}/signatures",
]:
    os.makedirs(d, exist_ok=True)

print(f"Directories ready")
print(f"   Input slides:  {DATA_DIR}/raw/slides/")
print(f"   Input RNA-seq: {DATA_DIR}/raw/rnaseq/")
print(f"   Output patches: {DATA_DIR}/processed/patches/")
print(f"   Output sigs:    {DATA_DIR}/signatures/")

Mounted at /content/drive
Directories ready
   Input slides:  /content/drive/MyDrive/ImmunoPath/data/raw/slides/
   Input RNA-seq: /content/drive/MyDrive/ImmunoPath/data/raw/rnaseq/
   Output patches: /content/drive/MyDrive/ImmunoPath/data/processed/patches/
   Output sigs:    /content/drive/MyDrive/ImmunoPath/data/signatures/


## Install dependancies


In [2]:

# OpenSlide needs system-level install on Linux (Colab)

import subprocess
subprocess.run(["apt-get", "install", "-y", "-qq", "openslide-tools"], check=True)
subprocess.run(["pip", "install", "-q", "openslide-python", "opencv-python-headless",
                 "Pillow", "numpy", "pandas", "scipy", "tqdm"], check=True)

# Verify imports
import openslide
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import stats
from tqdm.auto import tqdm
import json
from pathlib import Path
import glob

print(f"OpenSlide version: {openslide.__version__}")
print(f"All dependencies loaded")

OpenSlide version: 1.4.3
All dependencies loaded


## Reinhard Stain Normalization

In [3]:
# Reinhard normalization transfers color statistics in LAB space.
# This is MANDATORY per spec (NOT Macenko).
#
# Reference: Reinhard et al. "Color Transfer Between Images" (2001)
# We use fixed reference statistics from a representative TCGA H&E slide.

# Reference statistics (TCGA standard H&E)
# These are from a well-stained, high-quality TCGA lung adenocarcinoma slide
REF_MEANS = np.array([148.60, 169.30, 105.97])  # LAB channel means
REF_STDS  = np.array([ 41.56,   9.01,  14.67])  # LAB channel stds

def reinhard_normalize(image_rgb: np.ndarray,
                       ref_means: np.ndarray = REF_MEANS,
                       ref_stds: np.ndarray = REF_STDS) -> np.ndarray:
    """
    Apply Reinhard color normalization in LAB color space.

    Args:
        image_rgb: Input RGB image as numpy array (H, W, 3), dtype uint8
        ref_means: Reference LAB channel means [L, A, B]
        ref_stds: Reference LAB channel standard deviations [L, A, B]

    Returns:
        Normalized RGB image as numpy array (H, W, 3), dtype uint8
    """
    # Convert to LAB color space (float for precision)
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)

    # Compute source statistics
    src_means = np.mean(lab, axis=(0, 1))
    src_stds = np.std(lab, axis=(0, 1))

    # Transfer statistics channel by channel
    for i in range(3):
        if src_stds[i] > 1e-6:  # Avoid division by zero
            lab[:, :, i] = ((lab[:, :, i] - src_means[i])
                            * (ref_stds[i] / src_stds[i])
                            + ref_means[i])

    # Clip and convert back
    lab = np.clip(lab, 0, 255).astype(np.uint8)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def test_reinhard():
    """Quick visual test of Reinhard normalization."""
    # Create a synthetic pinkish image (simulating H&E)
    test_img = np.random.randint(150, 230, (256, 256, 3), dtype=np.uint8)
    test_img[:, :, 0] = np.clip(test_img[:, :, 0] + 30, 0, 255)  # More red
    test_img[:, :, 2] = np.clip(test_img[:, :, 2] + 20, 0, 255)  # More blue

    normalized = reinhard_normalize(test_img)

    # Check LAB stats of normalized image are closer to reference
    lab = cv2.cvtColor(normalized, cv2.COLOR_RGB2LAB).astype(np.float32)
    norm_means = np.mean(lab, axis=(0, 1))
    norm_stds = np.std(lab, axis=(0, 1))

    print("Reinhard normalization test:")
    print(f"  Reference means: {REF_MEANS}")
    print(f"  Normalized means: {norm_means}")
    print(f"  Reference stds:  {REF_STDS}")
    print(f"  Normalized stds: {norm_stds}")
    print(f" Normalization function works")

test_reinhard()

Reinhard normalization test:
  Reference means: [148.6  169.3  105.97]
  Normalized means: [147.84111 168.35928 105.25244]
  Reference stds:  [41.56  9.01 14.67]
  Normalized stds: [40.87998   9.448387 14.346277]
 Normalization function works


## Tissue Detection Functions

In [4]:

# Detect tissue regions in H&E patches using Otsu thresholding.
# Filter out background, pen marks, and artifacts.

def compute_tissue_mask(image_rgb: np.ndarray, threshold: int = 220) -> np.ndarray:
    """
    Create a binary tissue mask using grayscale thresholding.

    Tissue appears darker than background in H&E slides.
    Background is usually white/near-white (>220 in grayscale).

    Args:
        image_rgb: RGB image as numpy array
        threshold: Grayscale threshold (pixels < threshold = tissue)

    Returns:
        Binary mask (1 = tissue, 0 = background)
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    # Tissue is darker than background
    mask = (gray < threshold).astype(np.uint8)

    # Morphological cleanup (remove small holes and noise)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    return mask


def compute_tissue_fraction(image_rgb: np.ndarray, threshold: int = 220) -> float:
    """Compute fraction of tissue (non-background) pixels."""
    mask = compute_tissue_mask(image_rgb, threshold)
    return np.mean(mask)


def is_pen_mark(image_rgb: np.ndarray) -> bool:
    """
    Detect pen marks (common artifacts in TCGA slides).
    Pen marks are typically blue, green, or black ink.
    """
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)

    # Blue pen: high saturation, low value, hue in blue range
    blue_mask = ((hsv[:, :, 0] > 90) & (hsv[:, :, 0] < 130) &
                 (hsv[:, :, 1] > 50) & (hsv[:, :, 2] < 200))

    # Green pen
    green_mask = ((hsv[:, :, 0] > 35) & (hsv[:, :, 0] < 85) &
                  (hsv[:, :, 1] > 50) & (hsv[:, :, 2] < 200))

    # Black pen / very dark artifacts
    black_mask = (hsv[:, :, 2] < 40)

    pen_fraction = (np.sum(blue_mask) + np.sum(green_mask) + np.sum(black_mask)) / (image_rgb.shape[0] * image_rgb.shape[1])
    return pen_fraction > 0.1  # >10% pen pixels → reject


def is_blurry(image_rgb: np.ndarray, threshold: float = 50.0) -> bool:
    """Detect blurry patches using Laplacian variance."""
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian_var < threshold


print("Tissue detection functions defined")

Tissue detection functions defined


## WSI Patch Extraction Pipeline

In [5]:
# Extract 512×512 patches at 20× from .svs slides.
# Apply tissue detection, artifact filtering, and Reinhard normalization.

PATCH_SIZE = 512          # Pixels
TARGET_MPP = 0.5          # Microns per pixel (≈20× magnification)
TISSUE_THRESHOLD = 0.5    # Minimum tissue fraction per patch
MAX_PATCHES_PER_SLIDE = 64  # Memory-practical ceiling
JPEG_QUALITY = 95


def get_slide_mpp(slide: openslide.OpenSlide) -> float:
    """Get microns per pixel from slide properties."""
    try:
        mpp_x = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_X, 0))
        if mpp_x > 0:
            return mpp_x
    except (ValueError, KeyError):
        pass

    # Fallback: estimate from objective power
    try:
        power = float(slide.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER, 0))
        if power > 0:
            return 10.0 / power  # Rough conversion
    except (ValueError, KeyError):
        pass

    # Default assumption for TCGA slides
    return 0.25  # Most TCGA slides are scanned at 40× ≈ 0.25 mpp


def extract_patches_from_slide(
    slide_path: str,
    output_dir: str,
    patch_size: int = PATCH_SIZE,
    target_mpp: float = TARGET_MPP,
    tissue_threshold: float = TISSUE_THRESHOLD,
    max_patches: int = MAX_PATCHES_PER_SLIDE,
    normalize: bool = True,
    jpeg_quality: int = JPEG_QUALITY,
) -> dict:
    """
    Extract tissue patches from a whole-slide image.

    Args:
        slide_path: Path to .svs file
        output_dir: Directory to save patches
        patch_size: Patch dimensions in pixels
        target_mpp: Target microns per pixel
        tissue_threshold: Min tissue fraction
        max_patches: Max patches to extract
        normalize: Apply Reinhard normalization
        jpeg_quality: JPEG save quality

    Returns:
        Metadata dict with extraction stats
    """
    slide = openslide.OpenSlide(slide_path)
    slide_id = Path(slide_path).stem

    # Determine the best level for target_mpp
    slide_mpp = get_slide_mpp(slide)
    downsample_factor = target_mpp / slide_mpp if slide_mpp > 0 else 1.0

    # Find the closest available level
    best_level = 0
    for level in range(slide.level_count):
        level_ds = slide.level_downsamples[level]
        if level_ds <= downsample_factor * 1.2:  # Allow 20% tolerance
            best_level = level

    level_ds = slide.level_downsamples[best_level]
    level_dims = slide.level_dimensions[best_level]

    # Compute effective patch size at level 0 coordinates
    patch_size_l0 = int(patch_size * level_ds)

    # Create output directory
    patch_dir = os.path.join(output_dir, slide_id)
    os.makedirs(patch_dir, exist_ok=True)

    # Generate grid of patch locations
    width_l0, height_l0 = slide.dimensions
    patches_metadata = []
    all_candidates = []

    for y in range(0, height_l0 - patch_size_l0 + 1, patch_size_l0):
        for x in range(0, width_l0 - patch_size_l0 + 1, patch_size_l0):
            all_candidates.append((x, y))

    # Shuffle candidates for random sampling (when max_patches < total)
    np.random.shuffle(all_candidates)

    extracted = 0
    for x, y in tqdm(all_candidates, desc=f"{slide_id}", leave=False):
        if extracted >= max_patches:
            break

        try:
            # Read patch at best level
            patch = slide.read_region((x, y), best_level, (patch_size, patch_size))
            patch_rgb = np.array(patch.convert('RGB'))

            # Quality checks
            tissue_frac = compute_tissue_fraction(patch_rgb)
            if tissue_frac < tissue_threshold:
                continue

            if is_pen_mark(patch_rgb):
                continue

            if is_blurry(patch_rgb):
                continue

            # Apply Reinhard normalization
            if normalize:
                patch_rgb = reinhard_normalize(patch_rgb)

            # Save patch
            patch_name = f"{slide_id}_patch_{extracted:03d}.jpg"
            patch_path = os.path.join(patch_dir, patch_name)
            Image.fromarray(patch_rgb).save(patch_path, quality=jpeg_quality)

            patches_metadata.append({
                "patch_name": patch_name,
                "x": x,
                "y": y,
                "level": best_level,
                "tissue_fraction": round(tissue_frac, 3),
            })
            extracted += 1

        except Exception as e:
            continue  # Skip problematic patches

    slide.close()

    # Save slide metadata
    metadata = {
        "slide_id": slide_id,
        "slide_path": slide_path,
        "slide_mpp": slide_mpp,
        "extraction_level": best_level,
        "level_downsample": level_ds,
        "patch_size": patch_size,
        "target_mpp": target_mpp,
        "total_candidates": len(all_candidates),
        "patches_extracted": extracted,
        "normalized": normalize,
        "patches": patches_metadata,
    }

    meta_path = os.path.join(patch_dir, f"{slide_id}_metadata.json")
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    return metadata


print("Patch extraction pipeline defined")
print(f"   Settings: {PATCH_SIZE}×{PATCH_SIZE} at {TARGET_MPP} µm/px")
print(f"   Tissue threshold: {TISSUE_THRESHOLD}")
print(f"   Max patches/slide: {MAX_PATCHES_PER_SLIDE}")
print(f"   Reinhard normalization: enabled")

Patch extraction pipeline defined
   Settings: 512×512 at 0.5 µm/px
   Tissue threshold: 0.5
   Max patches/slide: 64
   Reinhard normalization: enabled


## Run Patch Extraction on Downloaded Slides

In [6]:
# Process all .svs slides from Phase 1 download.

slide_dir = f"{DATA_DIR}/raw/slides"
patch_output_dir = f"{DATA_DIR}/processed/patches"

# Find all .svs files
svs_files = glob.glob(os.path.join(slide_dir, "*.svs"))
print(f"Found {len(svs_files)} SVS slides to process")

if len(svs_files) == 0:
    print("\nNo .svs files found in slide directory!")
    print(f"   Expected location: {slide_dir}")
    print("   Run Phase 1 first to download slides.")
    print("   Or set MAX_SLIDES > 0 in Phase 1, Cell 6")
else:
    extraction_results = []

    for i, svs_path in enumerate(svs_files):
        slide_name = Path(svs_path).stem
        print(f"\n[{i+1}/{len(svs_files)}] Processing: {slide_name}")

        # Skip if already processed
        existing_meta = os.path.join(patch_output_dir, slide_name, f"{slide_name}_metadata.json")
        if os.path.exists(existing_meta):
            print(f"  Already processed — skipping")
            with open(existing_meta) as f:
                extraction_results.append(json.load(f))
            continue

        try:
            meta = extract_patches_from_slide(
                slide_path=svs_path,
                output_dir=patch_output_dir,
                patch_size=PATCH_SIZE,
                target_mpp=TARGET_MPP,
                tissue_threshold=TISSUE_THRESHOLD,
                max_patches=MAX_PATCHES_PER_SLIDE,
                normalize=True,
            )
            extraction_results.append(meta)
            print(f"   {meta['patches_extracted']} patches extracted "
                  f"(from {meta['total_candidates']} candidates)")
        except Exception as e:
            print(f"   Failed: {type(e).__name__}: {e}")

    # Summary
    total_patches = sum(r.get("patches_extracted", 0) for r in extraction_results)
    print(f"\n{'=' * 60}")
    print(f"EXTRACTION SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Slides processed: {len(extraction_results)}")
    print(f"  Total patches:    {total_patches}")
    print(f"  Output dir:       {patch_output_dir}")

Found 20 SVS slides to process

[1/20] Processing: TCGA-05-4425-01Z-00-DX1.82B093EE-49BC-4FD9-91AC-4CC89944309D


TCGA-05-4425-01Z-00-DX1.82B093EE-49BC-4FD9-91AC-4CC89944309D:   0%|          | 0/171 [00:00<?, ?it/s]

   64 patches extracted (from 171 candidates)

[2/20] Processing: TCGA-05-5715-01Z-00-DX1.D3F0A1FA-2507-45FF-823F-F9981E62BB4C


TCGA-05-5715-01Z-00-DX1.D3F0A1FA-2507-45FF-823F-F9981E62BB4C:   0%|          | 0/304 [00:00<?, ?it/s]

   64 patches extracted (from 304 candidates)

[3/20] Processing: TCGA-05-5429-01Z-00-DX1.20729065-FADA-4E43-98D7-AFA5FB4A0447


...

   64 patches extracted (from 5723 candidates)

[19/20] Processing: TCGA-56-5898-01Z-00-DX1.d539fc8b-ae31-4820-8c2e-5cf587958eb3


TCGA-56-5898-01Z-00-DX1.d539fc8b-ae31-4820-8c2e-5cf587958eb3:   0%|          | 0/2016 [00:00<?, ?it/s]

   64 patches extracted (from 2016 candidates)

[20/20] Processing: TCGA-99-7458-01Z-00-DX1.10ea0b2c-c763-40d1-83a4-3d4ae957fdb0


TCGA-99-7458-01Z-00-DX1.10ea0b2c-c763-40d1-83a4-3d4ae957fdb0:   0%|          | 0/3157 [00:00<?, ?it/s]

   64 patches extracted (from 3157 candidates)

EXTRACTION SUMMARY
  Slides processed: 20
  Total patches:    1216
  Output dir:       /content/drive/MyDrive/ImmunoPath/data/processed/patches


## Visualise Sample Patches (QC Check)

In [7]:

# Display some extracted patches to visually verify quality.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def show_sample_patches(patch_dir, n_patches=8):
    """Display random sample of extracted patches."""
    # Find all patch directories
    slide_dirs = [d for d in os.listdir(patch_dir)
                  if os.path.isdir(os.path.join(patch_dir, d))]

    if not slide_dirs:
        print("No patches to display")
        return

    # Collect patches
    all_patches = []
    for sd in slide_dirs:
        patches = glob.glob(os.path.join(patch_dir, sd, "*.jpg"))
        all_patches.extend(patches)

    if not all_patches:
        print("No patch files found")
        return

    # Sample and display
    n_show = min(n_patches, len(all_patches))
    indices = np.random.choice(len(all_patches), n_show, replace=False)

    fig, axes = plt.subplots(2, n_show // 2, figsize=(3 * (n_show // 2), 6))
    axes = axes.flatten()

    for idx, ax in zip(indices, axes):
        img = Image.open(all_patches[idx])
        ax.imshow(img)
        ax.set_title(Path(all_patches[idx]).stem[-10:], fontsize=8)
        ax.axis('off')

    plt.suptitle("Sample Extracted Patches (Reinhard Normalized)", fontsize=12)
    plt.tight_layout()

    qc_path = f"{PROJECT_DIR}/results/phase2_patch_samples.png"
    plt.savefig(qc_path, dpi=150)
    plt.show()
    print(f"QC image saved to: {qc_path}")

show_sample_patches(patch_output_dir)

QC image saved to: /content/drive/MyDrive/ImmunoPath/results/phase2_patch_samples.png


---
## Part B: Compute Immune Signatures from RNA-seq

This section processes GDC RNA-seq files to compute:
1. **Gene set signature scores** (CD8, IFNγ, TIL, PD-L1 related, exhaustion)
2. **CD274 expression** → median split → high/low
3. **Immune phenotype** classification
4. **Composite immune score**

Gene sets from IMMUNOPATH_LUNG_SPEC.md Section 5.3 Step 2.

## Define Immune Gene Sets

In [8]:

# From spec Section 5.3 Step 2 (EXACT gene lists — do not modify)

IMMUNE_GENE_SETS = {
    "cd8_signature": [
        "CD8A", "CD8B", "GZMA", "GZMB", "PRF1", "IFNG", "CXCL9", "CXCL10"
    ],
    "ifng_signature": [
        "IFNG", "STAT1", "CCR5", "CXCL9", "CXCL10", "CXCL11", "IDO1",
        "PRF1", "GZMA", "HLA-DRA", "CXCR6", "LAG3", "NKG7", "PSMB10",
        "CMKLR1", "CD8A", "TIGIT", "PDCD1LG2"
    ],
    "til_signature": [
        "CD3D", "CD3E", "CD3G", "CD4", "CD8A", "CD8B", "FOXP3",
        "CD19", "MS4A1", "NCAM1"  # MS4A1=CD20, NCAM1=CD56
    ],
    "pdl1_related": [
        "CD274", "PDCD1", "PDCD1LG2", "CTLA4", "LAG3", "TIGIT", "HAVCR2"
    ],
    "exhaustion_signature": [
        "PDCD1", "CTLA4", "LAG3", "TIGIT", "HAVCR2", "TOX", "ENTPD1"
    ],
}

# CD274 is the gene encoding PD-L1 protein
# We use CD274 mRNA as a PROXY for PD-L1 IHC (NOT a replacement)
CD274_GENE = "CD274"

# Gene alias mapping (common issues with TCGA gene names)
GENE_ALIASES = {
    "CD20": "MS4A1",    # CD20 is commonly called MS4A1 in RNA-seq
    "CD56": "NCAM1",    # CD56 is NCAM1
    "PD-L1": "CD274",   # PD-L1 protein, CD274 gene
    "PD-1": "PDCD1",    # PD-1 protein, PDCD1 gene
    "TIM-3": "HAVCR2",  # TIM-3 protein, HAVCR2 gene
}

print("Immune gene sets defined:")
for name, genes in IMMUNE_GENE_SETS.items():
    print(f"   {name}: {len(genes)} genes")

Immune gene sets defined:
   cd8_signature: 8 genes
   ifng_signature: 18 genes
   til_signature: 10 genes
   pdl1_related: 7 genes
   exhaustion_signature: 7 genes


## Load RNA Seq Data

In [11]:
import os, glob
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import pandas as pd
import numpy as np

# ---------- SAFE + FAST GDC RNA-seq LOADER ----------

USECOLS = [1, 6]  # gene_name, tpm_unstranded
DTYPES = {1: "string", 6: "float32"}

def is_gzip_file(path: str) -> bool:
    # check magic number instead of trusting .gz extension
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"

def load_gdc_rnaseq_fast(file_path: str) -> pd.Series:
    compression = "gzip" if is_gzip_file(file_path) else None

    df = pd.read_csv(
        file_path,
        sep="\t",
        compression=compression,
        usecols=USECOLS,
        dtype=DTYPES,
        engine="c",
        comment="#",   # handles files starting with "# "
    )

    df.columns = ["gene", "tpm"]

    # drop metadata rows
    df = df[~df["gene"].str.startswith("N_")]

    # handle duplicate gene names
    return df.groupby("gene", sort=False)["tpm"].max()


def _load_one(fpath):
    sid = Path(fpath).stem.replace(".tsv", "").replace(".gz", "")
    return sid, load_gdc_rnaseq_fast(fpath)


# ---------- LOAD ALL FILES (PARALLEL) ----------

rnaseq_dir = f"{DATA_DIR}/raw/rnaseq"
rnaseq_files = glob.glob(os.path.join(rnaseq_dir, "*.tsv*"))

print(f"Found {len(rnaseq_files)} RNA-seq files")
if not rnaseq_files:
    raise RuntimeError(f"No RNA-seq files found in {rnaseq_dir}")

patient_tpm = {}

with ProcessPoolExecutor(max_workers=8) as ex:
    for sid, tpm in tqdm(
        ex.map(_load_one, rnaseq_files),
        total=len(rnaseq_files),
        desc="Loading RNA-seq",
    ):
        patient_tpm[sid] = tpm


# ---------- BUILD EXPRESSION MATRIX ----------

expression_matrix = (
    pd.concat(patient_tpm, axis=1, copy=False)
      .T
      .astype("float32")
)

print(f"Expression matrix: {expression_matrix.shape[0]} samples × {expression_matrix.shape[1]} genes")


# ---------- LOG2 TRANSFORM ----------

expression_log2 = np.log2(expression_matrix + 1, dtype="float32")


# ---------- CHECK TARGET GENES ----------

target_genes = set().union(*IMMUNE_GENE_SETS.values())

found_genes = target_genes & set(expression_matrix.columns)
missing_genes = target_genes - found_genes

print(f"Target immune genes found: {len(found_genes)}/{len(target_genes)}")
if missing_genes:
    print("Missing genes:", sorted(missing_genes))


Found 948 RNA-seq files


Loading RNA-seq: 100%|██████████| 948/948 [00:24<00:00, 38.16it/s]


Expression matrix: 948 samples × 59427 genes
Target immune genes found: 34/34


## Compute Immune Signature Scores

In [12]:

# Method: Z-score each gene across all patients, then average within
# each gene set to get a single score per patient per signature.

def compute_signature_scores(expr_log2: pd.DataFrame,
                              gene_sets: dict) -> pd.DataFrame:
    """
    Compute immune signature scores from log2-TPM expression matrix.

    Method:
    1. Z-score normalize each gene across samples
    2. Average z-scores within each gene set
    3. Result: one score per patient per signature

    Args:
        expr_log2: DataFrame (patients × genes), log2(TPM+1)
        gene_sets: Dict mapping signature_name → list of gene symbols

    Returns:
        DataFrame (patients × signatures)
    """
    # Z-score all genes across patients
    z_scores = expr_log2.apply(stats.zscore, nan_policy='omit')

    scores = {}
    for sig_name, genes in gene_sets.items():
        # Find which genes are available
        available = [g for g in genes if g in z_scores.columns]

        if len(available) == 0:
            print(f"   {sig_name}: no genes found in expression data")
            scores[sig_name] = np.nan
            continue

        coverage = len(available) / len(genes)
        if coverage < 0.5:
            print(f"   {sig_name}: only {len(available)}/{len(genes)} genes found ({coverage:.0%})")

        # Average z-score of available genes
        scores[sig_name] = z_scores[available].mean(axis=1)

    return pd.DataFrame(scores)


if 'expression_log2' in dir():
    print("Computing immune signature scores...")
    sig_scores = compute_signature_scores(expression_log2, IMMUNE_GENE_SETS)
    print(f"\nSignature scores: {sig_scores.shape[0]} patients × {sig_scores.shape[1]} signatures")
    print(f"\nSummary statistics:")
    print(sig_scores.describe().round(3).to_string())
else:
    print("Expression matrix not available — skipping signature computation")

Computing immune signature scores...

Signature scores: 948 patients × 5 signatures

Summary statistics:
       cd8_signature  ifng_signature  til_signature  pdl1_related  exhaustion_signature
count        948.000         948.000        948.000       948.000               948.000
mean          -0.000          -0.000         -0.000        -0.000                 0.000
std            0.869           0.819          0.747         0.817                 0.774
min           -2.122          -2.131         -1.964        -2.057                -2.141
25%           -0.653          -0.552         -0.540        -0.572                -0.549
50%           -0.033          -0.038         -0.000        -0.007                 0.004
75%            0.577           0.570          0.543         0.575                 0.591
max            2.562           2.382          2.758         2.400                 2.291


## Compute CD274 expression labels


In [13]:

# CD274 (PD-L1 gene) expression: log2(TPM+1) → median split → high/low
# This is a SURROGATE for PD-L1 IHC protein expression (r²=0.65-0.81)

if 'expression_log2' in dir() and CD274_GENE in expression_log2.columns:
    cd274_log2 = expression_log2[CD274_GENE]
    cd274_median = cd274_log2.median()

    # Median split → binary labels
    cd274_label = pd.Series(
        np.where(cd274_log2 >= cd274_median, "high", "low"),
        index=cd274_log2.index,
        name="cd274_expression"
    )

    print(f"CD274 (PD-L1 gene) expression:")
    print(f"  Median log2(TPM+1): {cd274_median:.3f}")
    print(f"  High (≥ median):    {(cd274_label == 'high').sum()}")
    print(f"  Low  (< median):    {(cd274_label == 'low').sum()}")
    print(f"\n   REMINDER: This is RNA proxy, NOT IHC PD-L1 TPS.")
    print(f"     Use 'cd274_expression' (NOT 'pdl1_tps') in all code.")
else:
    print("CD274 gene not found — cannot compute PD-L1 proxy")
    cd274_log2 = None
    cd274_label = None

CD274 (PD-L1 gene) expression:
  Median log2(TPM+1): 3.154
  High (≥ median):    474
  Low  (< median):    474

   REMINDER: This is RNA proxy, NOT IHC PD-L1 TPS.
     Use 'cd274_expression' (NOT 'pdl1_tps') in all code.


## Classify Immune Phenotype

In [14]:

# Based on TIL and CD8 signature scores:
#   inflamed  = TIL high + CD8 high
#   excluded  = TIL high + CD8 low
#   desert    = TIL low
#
# Use median split on signature scores for high/low.

def classify_immune_phenotype(sig_scores: pd.DataFrame) -> pd.Series:
    """
    Classify immune phenotype from signature scores.

    Returns: Series with values 'inflamed', 'excluded', or 'desert'
    """
    til_score = sig_scores["til_signature"]
    cd8_score = sig_scores["cd8_signature"]

    til_median = til_score.median()
    cd8_median = cd8_score.median()

    til_high = til_score >= til_median
    cd8_high = cd8_score >= cd8_median

    phenotype = pd.Series("desert", index=sig_scores.index, name="immune_phenotype")
    phenotype[til_high & cd8_high] = "inflamed"
    phenotype[til_high & ~cd8_high] = "excluded"

    return phenotype


if 'sig_scores' in dir():
    immune_phenotype = classify_immune_phenotype(sig_scores)
    print(f"Immune phenotype distribution:")
    print(immune_phenotype.value_counts().to_string())
else:
    immune_phenotype = None

Immune phenotype distribution:
immune_phenotype
desert      474
inflamed    382
excluded     92


## Compute Composite Immune Score

In [15]:

# Average of (CD8 + IFNg + TIL scores) / 3, min-max normalized to [0, 1]

def compute_immune_score(sig_scores: pd.DataFrame) -> pd.Series:
    """Compute composite immune score, normalized to [0, 1]."""
    components = ["cd8_signature", "ifng_signature", "til_signature"]
    available = [c for c in components if c in sig_scores.columns]

    if not available:
        return pd.Series(np.nan, index=sig_scores.index, name="immune_score")

    raw_score = sig_scores[available].mean(axis=1)

    # Min-max normalize to [0, 1]
    score_min = raw_score.min()
    score_max = raw_score.max()

    if score_max - score_min > 0:
        normalized = (raw_score - score_min) / (score_max - score_min)
    else:
        normalized = pd.Series(0.5, index=raw_score.index)

    normalized.name = "immune_score"
    return normalized


if 'sig_scores' in dir():
    immune_score = compute_immune_score(sig_scores)
    print(f"Immune score summary (0 = cold, 1 = hot):")
    print(f"  Mean:   {immune_score.mean():.3f}")
    print(f"  Median: {immune_score.median():.3f}")
    print(f"  Std:    {immune_score.std():.3f}")
else:
    immune_score = None

Immune score summary (0 = cold, 1 = hot):
  Mean:   0.483
  Median: 0.477
  Std:    0.190


## Assemble final immune CSV

In [16]:

# Combine all computed features into one CSV.

if 'sig_scores' in dir():
    final_df = sig_scores.copy()
    final_df.index.name = "sample_id"

    # Add CD274 expression
    if cd274_log2 is not None:
        final_df["cd274_log2_tpm"] = cd274_log2
    if cd274_label is not None:
        final_df["cd274_expression"] = cd274_label

    # Add immune phenotype
    if immune_phenotype is not None:
        final_df["immune_phenotype"] = immune_phenotype

    # Add immune score
    if immune_score is not None:
        final_df["immune_score"] = immune_score

    # Save
    output_path = f"{DATA_DIR}/signatures/immune_signatures.csv"
    final_df.to_csv(output_path)

    print(f"Saved: {output_path}")
    print(f"   Shape: {final_df.shape}")
    print(f"   Columns: {list(final_df.columns)}")
    print(f"\n   Preview:")
    print(final_df.head().to_string())
else:
    print("No signature data to save. Run RNA-seq loading cells first.")

Saved: /content/drive/MyDrive/ImmunoPath/data/signatures/immune_signatures.csv
   Shape: (948, 9)
   Columns: ['cd8_signature', 'ifng_signature', 'til_signature', 'pdl1_related', 'exhaustion_signature', 'cd274_log2_tpm', 'cd274_expression', 'immune_phenotype', 'immune_score']

   Preview:
              cd8_signature  ifng_signature  til_signature  pdl1_related  exhaustion_signature  cd274_log2_tpm cd274_expression immune_phenotype  immune_score
sample_id                                                                                                                                                     
TCGA-44-6146      -1.356146       -1.252901      -1.515728     -1.746593             -1.637075        1.433654              low           desert      0.148421
TCGA-44-2665      -0.693099       -0.181803       0.056467      0.537460              0.741234        5.040393             high         excluded      0.416434
TCGA-44-2668       0.494382        0.465338      -0.504140      0.532003  

## Phase 2 summary

In [17]:


from datetime import datetime

print("=" * 60)
print("PHASE 2 — DATA PROCESSING SUMMARY")
print("=" * 60)

# Count patches
total_patches = 0
n_slides_processed = 0
if os.path.exists(patch_output_dir):
    for d in os.listdir(patch_output_dir):
        dpath = os.path.join(patch_output_dir, d)
        if os.path.isdir(dpath):
            n_patches = len(glob.glob(os.path.join(dpath, "*.jpg")))
            if n_patches > 0:
                n_slides_processed += 1
                total_patches += n_patches

print(f"\nPart A — Patch Extraction:")
print(f"  Slides processed:  {n_slides_processed}")
print(f"  Total patches:     {total_patches}")
print(f"  Patch size:        {PATCH_SIZE}×{PATCH_SIZE}")
print(f"  Target MPP:        {TARGET_MPP} (≈20×)")
print(f"  Normalization:     Reinhard (LAB color space)")
print(f"  Output:            {patch_output_dir}")

print(f"\nPart B — Immune Signatures:")
if 'final_df' in dir():
    print(f"  Patients:          {len(final_df)}")
    print(f"  Features:          {len(final_df.columns)}")
    if 'cd274_label' in dir() and cd274_label is not None:
        print(f"  CD274 high:        {(cd274_label == 'high').sum()}")
        print(f"  CD274 low:         {(cd274_label == 'low').sum()}")
    if 'immune_phenotype' in dir() and immune_phenotype is not None:
        print(f"  Inflamed:          {(immune_phenotype == 'inflamed').sum()}")
        print(f"  Excluded:          {(immune_phenotype == 'excluded').sum()}")
        print(f"  Desert:            {(immune_phenotype == 'desert').sum()}")
    print(f"  Output:            {DATA_DIR}/signatures/immune_signatures.csv")
else:
    print(f"  Not computed (no RNA-seq data)")

# Save phase report
report = {
    "phase": 2,
    "timestamp": datetime.now().isoformat(),
    "patches": {"slides": n_slides_processed, "total": total_patches,
                "size": PATCH_SIZE, "mpp": TARGET_MPP, "normalization": "reinhard"},
    "signatures": {"patients": len(final_df) if 'final_df' in dir() else 0,
                    "features": list(final_df.columns) if 'final_df' in dir() else []},
}
report_path = f"{PROJECT_DIR}/results/phase2_processing_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f"\n Report: {report_path}")

print(f"\n{'=' * 60}")
print(" UPDATE PHASE_TRACKER.md:")
print(f"{'=' * 60}")
print(f"  Status:                    DONE")
print(f"  Patches Extracted:         {total_patches} from {n_slides_processed} slides")
print(f"  Reinhard Normalization:    Applied")
print(f"  Immune Signatures:         {'Yes' if 'final_df' in dir() else 'No'}")
print(f"  immune_signatures.csv:     {len(final_df) if 'final_df' in dir() else 0} rows")

print(f"\n{'=' * 60}")
print("NEXT: Phase 3 — Training Data Creation")
print(f"{'=' * 60}")
print("1. Join patches with immune signatures by patient ID")
print("2. Add Bagaev TME subtypes + MSI labels")
print("3. Create JSONL training files (train/val/test)")
print("4. Apply PATIENT-LEVEL splits (prevent leakage)")

PHASE 2 — DATA PROCESSING SUMMARY

Part A — Patch Extraction:
  Slides processed:  19
  Total patches:     1216
  Patch size:        512×512
  Target MPP:        0.5 (≈20×)
  Normalization:     Reinhard (LAB color space)
  Output:            /content/drive/MyDrive/ImmunoPath/data/processed/patches

Part B — Immune Signatures:
  Patients:          948
  Features:          9
  CD274 high:        474
  CD274 low:         474
  Inflamed:          382
  Excluded:          92
  Desert:            474
  Output:            /content/drive/MyDrive/ImmunoPath/data/signatures/immune_signatures.csv

 Report: /content/drive/MyDrive/ImmunoPath/results/phase2_processing_report.json

 UPDATE PHASE_TRACKER.md:
  Status:                    DONE
  Patches Extracted:         1216 from 19 slides
  Reinhard Normalization:    Applied
  Immune Signatures:         Yes
  immune_signatures.csv:     948 rows

NEXT: Phase 3 — Training Data Creation
1. Join patches with immune signatures by patient ID
2. Add Bagaev 